# DimUser

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("parquet")\
          .load("abfss://bronze@spotifydatalakepro.dfs.core.windows.net/DimUser")

df.display()          

user_id,user_name,country,subscription_type,start_date,end_date,updated_at
1,Carlos Berry,Switzerland,Premium,2023-10-17,null,2025-09-23T19:49:55.000Z
2,Amanda Jenkins,Montserrat,Family,2024-09-28,null,2025-09-29T19:49:55.000Z
3,Daniel Cook,Chile,Premium,2025-07-07,null,2025-09-08T19:49:55.000Z
4,Peter Hernandez,Nigeria,Free,2024-07-16,null,2025-09-29T19:49:55.000Z
5,Yolanda Morris,Aruba,Premium,2025-05-12,null,2025-09-17T19:49:55.000Z
6,Stephen Murphy,Cook Islands,Free,2025-06-17,null,2025-09-12T19:49:55.000Z
7,Anthony Andrews Jr.,Libyan Arab Jamahiriya,Free,2024-06-08,null,2025-09-20T19:49:55.000Z
8,Felicia Jones,Haiti,Family,2024-09-20,null,2025-09-08T19:49:55.000Z
9,Hector Decker,Palau,Family,2024-01-08,null,2025-09-29T19:49:55.000Z
10,Frank Davis,New Zealand,Family,2023-12-13,null,2025-09-22T19:49:55.000Z


## Autoloader

In [0]:
df_user = spark.readStream.format("cloudFiles")\
              .option("cloudFiles.format","parquet")\
              .option("cloudFiles.schemaLocation","abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimUser/checkpoint")\
              .load("abfss://bronze@spotifydatalakepro.dfs.core.windows.net/DimUser")

In [0]:
display(df_user, checkpointLocation="abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimUser/checkpoint")

user_id,user_name,country,subscription_type,start_date,end_date,updated_at,_rescued_data
1,Carlos Berry,Switzerland,Premium,2023-10-17,null,2025-09-23T19:49:55.000Z,null
2,Amanda Jenkins,Montserrat,Family,2024-09-28,null,2025-09-29T19:49:55.000Z,null
3,Daniel Cook,Chile,Premium,2025-07-07,null,2025-09-08T19:49:55.000Z,null
4,Peter Hernandez,Nigeria,Free,2024-07-16,null,2025-09-29T19:49:55.000Z,null
5,Yolanda Morris,Aruba,Premium,2025-05-12,null,2025-09-17T19:49:55.000Z,null
6,Stephen Murphy,Cook Islands,Free,2025-06-17,null,2025-09-12T19:49:55.000Z,null
7,Anthony Andrews Jr.,Libyan Arab Jamahiriya,Free,2024-06-08,null,2025-09-20T19:49:55.000Z,null
8,Felicia Jones,Haiti,Family,2024-09-20,null,2025-09-08T19:49:55.000Z,null
9,Hector Decker,Palau,Family,2024-01-08,null,2025-09-29T19:49:55.000Z,null
10,Frank Davis,New Zealand,Family,2023-12-13,null,2025-09-22T19:49:55.000Z,null


In [0]:
df_user = df_user.withColumn("user_name",upper(col("user_name")))


In [0]:
import os
import sys

# Add the parent directory of spotify_dab to sys.path
notebook_path = "/Workspace/Users/surajlk9755@gmail.com/SpotifyAzureProject/spotify_dab/src/silver"
sys.path.insert(0, notebook_path)

# Import directly from the file
import importlib.util
spec = importlib.util.spec_from_file_location("transformations", 
    os.path.join(notebook_path, "spotify_dab/utils/transformations.py"))
transformations = importlib.util.module_from_spec(spec)
spec.loader.exec_module(transformations)

var_surak = transformations.var_surak

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-5974410421484578>, line 15
     12 transformations = importlib.util.module_from_spec(spec)
     13 spec.loader.exec_module(transformations)
---> 15 var_surak = transformations.var_surak

AttributeError: module 'transformations' has no attribute 'var_surak'

In [0]:
import os
import sys

project_path = os.path.join(os.getcwd(),"..","..")
sys.path.append(project_path)

from spotify_dab.utils.transformations import reusable

In [0]:
df_user_obj = reusable()


In [0]:
df_user = df_user_obj.dropColumn(df_user,['_rescued_data'])
#display(df_user,checkpointLocation = "abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimUser/checkpoint")



In [0]:
df_user = df_user.dropDuplicates(['user_id'])

In [0]:
df_user.writeStream.format("delta")\
       .outputMode("append")\
       .option("checkpointLocation","abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimUser/checkpoint_v2")\
       .trigger(once=True)\
       .start("abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimUser/data")  

In [0]:
from pyspark.sql.functions import upper

# Set correct partition count
spark.conf.set("spark.sql.shuffle.partitions", 200)

# Clean transformation (ONLY ONE dedup)
df_user_clean = (
    df_user
        .select(
            "user_id",
            upper("user_name").alias("user_name"),
            "country",
            "subscription_type",
            "start_date",
            "end_date",
            "updated_at"
        )
        
        # Partition BEFORE stateful op
        .repartition("user_id")
        
        # Single stateful operation
        .dropDuplicates(["user_id"])
)

# 🚨 NEW checkpoint (mandatory now)
query = (
    df_user_clean.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimUser/checkpoint_v3"
        )
        .trigger(once=True)\
        .option("path", "abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimUser/data")    
        .table("spotify_cata.silver.DimUser")
)

query.awaitTermination()

26/05/02 15:57:18 Query fe471f92-c770-44f7-a236-73236e8b1296 have exception: org.apache.spark.SparkRuntimeException: [STREAMING_STATEFUL_OPERATOR_NOT_MATCH_IN_STATE_METADATA] Streaming stateful operator name does not match with the operator in state metadata. This likely to happen when user adds/removes/changes stateful operator of existing streaming query.
Stateful operators in the metadata: [(OperatorId: 0 -> OperatorName: dedupe), (OperatorId: 1 -> OperatorName: dedupe)]; Stateful operators in current batch: [(OperatorId: 0 -> OperatorName: dedupe), (OperatorId: 1 -> OperatorName: dedupe), (OperatorId: 2 -> OperatorName: dedupe)]. SQLSTATE: 42K03
	at org.apache.spark.sql.errors.QueryExecutionErrors$.statefulOperatorNotMatchInStateMetadataError(QueryExecutionErrors.scala:2601)
	at org.apache.spark.sql.execution.streaming.runtime.IncrementalExecution$$anon$2.$anonfun$checkOperatorValidWithMetadata$3(IncrementalExecution.scala:998)
	at scala.runtime.java8.JFunction1$mcVJ$sp.apply(JFunc

---------------------------------------------------------------------------
StreamingQueryException                   Traceback (most recent call last)
File <command-5974410421484742>, line 40
     26 # 🚨 NEW checkpoint (mandatory now)
     27 query = (
     28     df_user_clean.writeStream
     29         .format("delta")
   (...)
     37         .table("spotify_cata.silver.DimUser")
     38 )
---> 40 query.awaitTermination()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/query.py:100, in StreamingQuery.awaitTermination(self, timeout)
     98 await_termination_cmd = pb2.StreamingQueryCommand.AwaitTerminationCommand()
     99 cmd.await_termination.CopyFrom(await_termination_cmd)
--> 100 self._execute_streaming_query_cmd(cmd)
    101 return None

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/query.py:188, in StreamingQuery._execute_streaming_query_cmd(self, cmd)
    186 exec_cmd = pb2.Command()
    187 exec_cmd.st

# DimArtist

In [0]:
df = spark.read.format("parquet")\
          .load("abfss://bronze@spotifydatalakepro.dfs.core.windows.net/DimArtist")
df.display()


artist_id,artist_name,genre,country,updated_at
4,Anne Rose,Classical,Timor-Leste,2025-10-07T19:49:56.000Z
1,Lauren Singleton,Rock,Oman,2025-09-26T19:49:55.000Z
2,Gregory Mccarty,Classical,Suriname,2025-09-23T19:49:55.000Z
3,Adam Stanton,Hip-Hop,Anguilla,2025-09-09T19:49:55.000Z
4,Monica Roth,Jazz,Argentina,2025-09-15T19:49:55.000Z
5,Calvin Blake,Rock,Greenland,2025-09-18T19:49:55.000Z
6,James Lowe,Pop,Saudi Arabia,2025-09-17T19:49:55.000Z
7,Jacqueline Evans,Jazz,Syrian Arab Republic,2025-09-10T19:49:55.000Z
8,Paul Johnson,Electronic,Bangladesh,2025-09-19T19:49:55.000Z
9,Alexa Garcia,Rock,Burkina Faso,2025-09-09T19:49:55.000Z


In [0]:
df_art = spark.readStream.format("cloudFiles")\
              .option("cloudFiles.format","parquet")\
              .option("cloudFiles.schemaLocation","abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimArt/checkpoint")\
              .option("schemaEvolutionMode","addNewColumns")\
              .load("abfss://bronze@spotifydatalakepro.dfs.core.windows.net/DimArtist")

In [0]:
display(df_art,checkpointLocation = "abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimArt/checkpoint")



artist_id,artist_name,genre,country,updated_at,_rescued_data
1,Lauren Singleton,Rock,Oman,2025-09-26T19:49:55.000Z,null
2,Gregory Mccarty,Classical,Suriname,2025-09-23T19:49:55.000Z,null
3,Adam Stanton,Hip-Hop,Anguilla,2025-09-09T19:49:55.000Z,null
4,Monica Roth,Jazz,Argentina,2025-09-15T19:49:55.000Z,null
5,Calvin Blake,Rock,Greenland,2025-09-18T19:49:55.000Z,null
6,James Lowe,Pop,Saudi Arabia,2025-09-17T19:49:55.000Z,null
7,Jacqueline Evans,Jazz,Syrian Arab Republic,2025-09-10T19:49:55.000Z,null
8,Paul Johnson,Electronic,Bangladesh,2025-09-19T19:49:55.000Z,null
9,Alexa Garcia,Rock,Burkina Faso,2025-09-09T19:49:55.000Z,null
10,Robert Ellis,Rock,Ecuador,2025-09-19T19:49:55.000Z,null


In [0]:
df_art_obj = reusable()
df_art = df_art_obj.dropColumn(df_art,['_rescued_data'])


In [0]:
df_art = df_art.dropDuplicates(['artist_id'])
#display(df_art,checkpointLocation = "abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimArt/checkpoint")



In [0]:
df_art.writeStream.format("delta")\
       .outputMode("append")\
       .option("checkpointLocation","abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimArt/checkpoint_v2")\
       .trigger(once=True)\
       .start("abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimArt/data")  

In [0]:
from pyspark.sql.functions import *

# Ensure consistent partition count
spark.conf.set("spark.sql.shuffle.partitions", 200)

# Apply transformations (example structure — keep yours if already defined)
df_art_clean = (
    df_art
        # ✅ Partition BEFORE any stateful operation
        .repartition(200)
)

# ✅ Write stream with NEW checkpoint (fixes the error definitively)
query = (
    df_art_clean.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimArt/checkpoint_v3"
        )
        .trigger(once=True)
        .option("path","abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimArt/data")\
        .table("spotify_cata.silver.DimArtist"))

query.awaitTermination()

# DimTrack

In [0]:
df = spark.read.format("parquet")\
          .load("abfss://bronze@spotifydatalakepro.dfs.core.windows.net/DimTrack")
df.display()


track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at
1,Stand-alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z
2,Programmable contextually-based forecast,418,Current Album,110,2023-09-30,2025-09-26T19:49:55.000Z
3,Enhanced tertiary Internet solution,35,Born Album,195,2022-01-11,2025-09-11T19:49:55.000Z
4,Mandatory user-facing framework,54,Ball Album,167,2023-06-30,2025-10-02T19:49:55.000Z
5,Multi-layered needs-based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z
6,Synchronized high-level complexity,384,City Album,250,2021-10-16,2025-09-22T19:49:55.000Z
7,Total human-resource structure,317,Parent Album,194,2022-08-02,2025-10-07T19:49:55.000Z
8,Exclusive multimedia matrices,37,Owner Album,206,2022-06-22,2025-10-02T19:49:55.000Z
9,Digitized multimedia hardware,165,Hand Album,342,2023-12-03,2025-10-02T19:49:55.000Z
10,Phased dynamic utilization,15,Hair Album,199,2023-12-17,2025-10-07T19:49:55.000Z


In [0]:
df_track = spark.readStream.format("cloudFiles")\
              .option("cloudFiles.format","parquet")\
              .option("cloudFiles.schemaLocation","abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimTrack/checkpoint")\
              .option("schemaEvolutionMode","addNewColumns")\
              .load("abfss://bronze@spotifydatalakepro.dfs.core.windows.net/DimTrack")

In [0]:
display(df_track,checkpointLocation = "abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimTrack/checkpoint")




track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,_rescued_data
1,Stand-alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z,null
2,Programmable contextually-based forecast,418,Current Album,110,2023-09-30,2025-09-26T19:49:55.000Z,null
3,Enhanced tertiary Internet solution,35,Born Album,195,2022-01-11,2025-09-11T19:49:55.000Z,null
4,Mandatory user-facing framework,54,Ball Album,167,2023-06-30,2025-10-02T19:49:55.000Z,null
5,Multi-layered needs-based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z,null
6,Synchronized high-level complexity,384,City Album,250,2021-10-16,2025-09-22T19:49:55.000Z,null
7,Total human-resource structure,317,Parent Album,194,2022-08-02,2025-10-07T19:49:55.000Z,null
8,Exclusive multimedia matrices,37,Owner Album,206,2022-06-22,2025-10-02T19:49:55.000Z,null
9,Digitized multimedia hardware,165,Hand Album,342,2023-12-03,2025-10-02T19:49:55.000Z,null
10,Phased dynamic utilization,15,Hair Album,199,2023-12-17,2025-10-07T19:49:55.000Z,null


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
from pyspark.sql.functions import when, col

df_track = df_track.withColumn("durationFlag",when(col('duration_sec')<150, "low")\
                               .when(col('duration_sec')<300, "medium").otherwise("high"))
                                
                           



In [0]:
df_track = df_track.withColumn("track_name", regexp_replace(col("track_name"),"-"," "))

In [0]:
df_track = reusable().dropColumn(df_track, ['_rescued_data'])

In [0]:
df_track.writeStream.format("delta")\
       .outputMode("append")\
       .option("checkpointLocation","abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimTrack/checkpoint_v2")\
       .trigger(once=True)\
       .option("path","abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimTrack/data")\
       .table("spotify_cata.silver.DimTrack")

In [0]:
# Ensure consistent partitioning
spark.conf.set("spark.sql.shuffle.partitions", 200)

query = (
    df_track
        .repartition(200)  # safe for stateful ops
        .writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimTrack/checkpoint_v2"
        )  # use new checkpoint to avoid old state issues
        .trigger(once=True)
        .toTable("spotify_cata.silver.DimTrack")  # correct table sink
)

query.awaitTermination()

In [0]:
df_track = spark.readStream.format("cloudFiles")\
              .option("cloudFiles.format","parquet")\
              .option("cloudFiles.schemaLocation","abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimTrack/checkpoint_v2")\
              .option("schemaEvolutionMode","addNewColumns")\
              .load("abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimTrack/data")

In [0]:
display(spark.read.format("delta").load("abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimTrack/data"), checkpointLocation = "abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimTrack/checkpoint_v2")

track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,durationFlag
158,Fully configurable clear thinking help desk,71,Light Album,112,2023-10-04,2025-09-24T19:49:55.000Z,low
252,Compatible fault tolerant data warehouse,21,Child Album,256,2021-11-12,2025-09-24T19:49:55.000Z,medium
135,Assimilated web enabled focus group,24,Your Album,249,2022-07-19,2025-09-27T19:49:55.000Z,medium
198,Open architected stable attitude,338,Bring Album,231,2021-12-22,2025-09-13T19:49:55.000Z,medium
70,Face to face transitional Graphical User Interface,34,Get Album,98,2022-01-01,2025-09-24T19:49:55.000Z,low
42,Visionary client driven task force,178,Attention Album,276,2022-12-06,2025-10-01T19:49:55.000Z,medium
182,Customer focused radical conglomeration,374,Drop Album,357,2024-05-09,2025-10-05T19:49:55.000Z,high
244,Implemented dynamic secured line,395,Candidate Album,330,2022-09-04,2025-10-01T19:49:55.000Z,high
114,Managed bi directional pricing structure,373,Letter Album,180,2025-05-17,2025-09-13T19:49:55.000Z,medium
412,Future proofed composite utilization,427,Fact Album,202,2023-03-23,2025-09-13T19:49:55.000Z,medium


# DimDate

In [0]:
df_date = spark.readStream.format("cloudFiles")\
              .option("cloudFiles.format","parquet")\
              .option("cloudFiles.schemaLocation","abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimDate/checkpoint")\
              .option("schemaEvolutionMode","addNewColumns")\
              .load("abfss://bronze@spotifydatalakepro.dfs.core.windows.net/DimDate")

In [0]:
df_date = reusable().dropColumn(df_date, ['_rescued_data'])


In [0]:
df_date.writeStream.format("delta")\
       .outputMode("append")\
       .option("checkpointLocation","abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimDate/checkpoint_v3")\
       .trigger(once=True)\
       .option("path","abfss://silver@spotifydatalakepro.dfs.core.windows.net/DimDate/data")\
       .table("spotify_cata.silver.DimDate")

# FactStream

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
              .option("cloudFiles.format","parquet")\
              .option("cloudFiles.schemaLocation","abfss://silver@spotifydatalakepro.dfs.core.windows.net/FactStream/checkpoint")\
              .option("schemaEvolutionMode","addNewColumns")\
              .load("abfss://bronze@spotifydatalakepro.dfs.core.windows.net/FactStream")

In [0]:
df_fact = reusable().dropColumn(df_fact, ['_rescued_data'])


In [0]:
display(df_fact, checkpointLocation = "abfss://silver@spotifydatalakepro.dfs.core.windows.net/FactStream/checkpoint_v2")


Checkpointing to abfss://silver@spotifydatalakepro.dfs.core.windows.net/FactStream/checkpoint_v2


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5974410421484775>, line 1
----> 1 display(df_fact, checkpointLocation = "abfss://silver@spotifydatalakepro.dfs.core.windows.net/FactStream/checkpoint_v2")

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:97, in Display.display_connect_table(self, df, **kwargs)
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was missing data, please restart the notebook or contact Databricks support"
     95       ) from e
     96 if df.isStream

In [0]:
df_fact.writeStream.format("delta")\
       .outputMode("append")\
       .option("checkpointLocation","abfss://silver@spotifydatalakepro.dfs.core.windows.net/FactStream/checkpoint_v3")\
       .trigger(once=True)\
       .option("path","abfss://silver@spotifydatalakepro.dfs.core.windows.net/FactStream/data")\
       .table("spotify_cata.silver.FactStream")